# Bankruptcy cascade — Python replica of `judge_bankrupt.R` + `simulate_bankrupt.R`

Exact port of the R contagion model (`financial-contagion-in-R/`) for speed.

**Model (threshold default contagion):** starting from one failed bank, a healthy bank `i` fails when the
total weight of incoming edges from already-failed banks exceeds `Ki` (0.04; 0.10 for high-degree nodes
under `target_policy`). The cascade propagates along out-edges until no new failures occur.

This notebook (1) validates the port against the R-produced `*_nodes_with_cascade.csv` for the smaller
networks, then (2) runs the per-node cascade for **`network_er_avgdeg_2_7_edges.csv`**.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
R_DIR = PROJECT_ROOT / 'financial-contagion-in-R'

NETWORK_SIZE = 10000   # as in ER_10.R (graph has 10000 nodes, ids 1..N)
KI_BASE      = 0.04    # Ki in judge_bankrupt.R
KI_TARGET    = 0.10    # Ki for high-degree nodes when target_policy=True
print('R dir:', R_DIR)

R dir: /Users/rubenmarques/Documents/Repositórios/Thesis/financial-contagion-in-R


## Build graph from an edges CSV

Mirrors igraph: directed edges `from -> to` with `weight`; vertices are ids `1..network_size`.
`out_nb[i]` = successors (i->j); `in_w[i]` = {j: weight} for predecessors (j->i, used by the threshold).

In [2]:
def build_graph(edges_csv, network_size=NETWORK_SIZE):
    e = pd.read_csv(edges_csv)
    n = max(int(network_size), int(e[['from', 'to']].max().max()))
    out_nb = {i: [] for i in range(1, n + 1)}
    in_w   = {i: {} for i in range(1, n + 1)}
    deg_in  = np.zeros(n + 1); deg_out = np.zeros(n + 1)
    str_in  = np.zeros(n + 1); str_out = np.zeros(n + 1)
    for f, t, w in zip(e['from'].astype(int), e['to'].astype(int), e['weight'].astype(float)):
        out_nb[f].append(t)
        in_w[t][f] = w          # simple graph: one edge per (f,t)
        deg_out[f] += 1; deg_in[t] += 1
        str_out[f] += w; str_in[t] += w
    deg_total = deg_in + deg_out
    g = dict(n=n, out_nb=out_nb, in_w=in_w,
             deg_in=deg_in, deg_out=deg_out, deg_total=deg_total,
             str_in=str_in, str_out=str_out, str_total=str_in + str_out)
    # 95th percentile of total degree (only used when target_policy=True)
    g['deg_q95'] = float(np.quantile(deg_total[1:n + 1], 0.95))
    return g

## `judge_bankrupt` + `simulate_bankrupt` (exact port)

In [3]:
def judge_bankrupt(g, this_batch_neighbor, already_bankrupt, target_policy=False):
    in_w, deg_total, q95 = g['in_w'], g['deg_total'], g['deg_q95']
    Ki = KI_BASE
    bankrupt = []
    for i in this_batch_neighbor:
        if target_policy and deg_total[i] >= q95:
            Ki = KI_TARGET   # NB: as in the R code, Ki is not reset per-iteration
        s = 0.0; has_bankrupt_in = False
        for j, w in in_w[i].items():
            if j in already_bankrupt:
                s += w; has_bankrupt_in = True
        if has_bankrupt_in and s > Ki:
            bankrupt.append(i)
    return set(bankrupt)


def simulate_bankrupt(g, method='random', type='banks', target_policy=False,
                      initial_node=None, rng=None):
    out_nb, deg_total, n = g['out_nb'], g['deg_total'], g['n']
    if method == 'biggest':
        initial = int(np.argmax(deg_total[1:n + 1])) + 1   # first node with max total degree
    elif method == 'node':
        if initial_node is None:
            raise ValueError("initial_node required when method='node'")
        initial = int(initial_node)
    else:
        initial = (rng or random).randint(1, n)

    bankrupt = {initial}
    this_batch = [initial]
    number_bankrupt = 1
    while True:
        cand = set()
        for b in this_batch:
            cand.update(out_nb[b])
        new_b = judge_bankrupt(g, cand, bankrupt, target_policy=target_policy)
        bankrupt |= new_b
        if len(bankrupt) == number_bankrupt:
            break
        number_bankrupt = len(bankrupt)
        this_batch = list(new_b)
    return len(bankrupt) if type == 'num' else sorted(bankrupt)

## Per-node cascade driver (mirrors `ER_10.R::main_per_node`)

In [4]:
def compute_node_cascades(edges_csv, avg_degree, network_size=NETWORK_SIZE, target_policy=False):
    g = build_graph(edges_csv, network_size)
    n = g['n']
    cascade = np.ones(n + 1, dtype=int)
    for node in range(1, n + 1):
        cascade[node] = simulate_bankrupt(g, method='node', initial_node=node,
                                          type='num', target_policy=target_policy)
    node_df = pd.DataFrame({
        'avg_degree': avg_degree,
        'node_id': np.arange(1, n + 1),
        'degree_in': g['deg_in'][1:n + 1].astype(int),
        'degree_out': g['deg_out'][1:n + 1].astype(int),
        'degree_total': g['deg_total'][1:n + 1].astype(int),
        'strength_in': g['str_in'][1:n + 1],
        'strength_out': g['str_out'][1:n + 1],
        'strength_total': g['str_total'][1:n + 1],
        'cascade_size': cascade[1:n + 1],
    })
    node_df['cascade_percentage'] = node_df['cascade_size'] / network_size
    return node_df

## (1) Validate against R results on the smaller networks

Recompute cascades in Python and compare `cascade_size` to the R-produced `*_nodes_with_cascade.csv`.

In [5]:
for label, avg in [('0_2', 0.2), ('0_4', 0.4), ('0_6', 0.6), ('0_8', 0.8)]:
    ref_path = R_DIR / f'network_er_avgdeg_{label}_nodes_with_cascade.csv'
    edges_path = R_DIR / f'network_er_avgdeg_{label}_edges.csv'
    if not ref_path.exists() or not edges_path.exists():
        print(f'{label}: reference/edges missing, skipping'); continue
    ref = pd.read_csv(ref_path)
    got = compute_node_cascades(edges_path, avg, network_size=NETWORK_SIZE)
    merged = ref[['node_id', 'cascade_size']].merge(
        got[['node_id', 'cascade_size']], on='node_id', suffixes=('_R', '_py'))
    n_match = (merged['cascade_size_R'] == merged['cascade_size_py']).sum()
    print(f'{label}: {n_match}/{len(merged)} nodes match  '
          f"(max|diff|={ (merged['cascade_size_R']-merged['cascade_size_py']).abs().max() })")

0_2: 10000/10000 nodes match  (max|diff|=0)
0_4: 10000/10000 nodes match  (max|diff|=0)
0_6: 10000/10000 nodes match  (max|diff|=0)


0_8: 10000/10000 nodes match  (max|diff|=0)


## (2) Run on `network_er_avgdeg_2_7_edges.csv` and save

In [6]:
EDGES_27 = R_DIR / 'network_er_avgdeg_2_7_edges.csv'
node_df_27 = compute_node_cascades(EDGES_27, avg_degree=2.7, network_size=NETWORK_SIZE)
out_path = R_DIR / 'network_er_avgdeg_2_7_nodes_with_cascade.csv'
node_df_27.to_csv(out_path, index=False)
print('Saved:', out_path)
print('cascade_size  min/mean/max =', node_df_27['cascade_size'].min(),
      round(node_df_27['cascade_size'].mean(), 3), node_df_27['cascade_size'].max())
node_df_27.head()

Saved: /Users/rubenmarques/Documents/Repositórios/Thesis/financial-contagion-in-R/network_er_avgdeg_2_7_nodes_with_cascade.csv
cascade_size  min/mean/max = 1 7087.089 9131


,avg_degree,node_id,degree_in,degree_out,degree_total,strength_in,strength_out,strength_total,cascade_size,cascade_percentage
0,2.7,1,0,5,5,0.0,0.306667,0.306667,9126,0.9126
1,2.7,2,2,1,3,0.2,0.200000,0.400000,9124,0.9124
2,2.7,3,6,4,10,0.2,0.246667,0.446667,9124,0.9124
3,2.7,4,2,2,4,0.2,0.150000,0.350000,9124,0.9124
4,2.7,5,3,5,8,0.2,0.323333,0.523333,9124,0.9124
